# Canonical MobileNetV2 Preprocessing Sensitivity Experiment

## RGB vs Grayscale × Frozen vs Fine-tuned × 5 Frozen Group-Aware Splits

Notebook ini menjalankan **hanya 20 run tambahan** dengan preprocessing canonical MobileNetV2. Eksperimen asli `[0,1]` tidak ditimpa.

### Faktor baru

Eksperimen asli:

`x / 255.0 -> [0,1]`

Eksperimen sensitivitas ini:

`tf.keras.applications.mobilenet_v2.preprocess_input(x) -> sekitar [-1,1]`

### Faktor yang dipertahankan identik

- split CSV asli yang sudah dibekukan;
- split seeds `42, 123, 7, 2024, 99`;
- fixed training seed `42`;
- MobileNetV2 ImageNet;
- input `224×224×3`;
- batch 32;
- augmentasi yang sama;
- bobot grayscale luminance yang sama;
- learning rate dan classifier head yang sama;
- fine-tuning mulai layer index 100;
- BatchNorm backbone tetap inference mode;
- max 50 epoch, patience 5.

Desain tambahan:

**2 representasi × 2 strategi training × 5 split = 20 CNN baru**

Run ID baru memakai prefix `exp5_canonical_...`.

Semua hasil ilmiah disimpan permanen di:

`MyDrive/malaria_colorspace/canonical_preprocessing/`

Data citra diekstrak ke `/content/data` hanya sebagai cache lokal cepat dan akan dibangun ulang otomatis setelah reconnect.

## Kebijakan disconnect / reconnect Colab

Jika runtime terputus:

1. reconnect ke GPU;
2. buka notebook ini;
3. jalankan dari atas;
4. Drive di-mount kembali;
5. dataset lokal diekstrak ulang bila `/content/data` hilang;
6. SHA-256 kelima split asli diverifikasi lagi;
7. jalankan loop eksperimen;
8. run yang sudah lengkap otomatis `SKIP`;
9. run yang terputus mencoba resume dari `BackupAndRestore` di Drive;
10. fallback tersedia dari `latest.weights.h5 + history.csv`;
11. jika training sudah selesai tetapi evaluation belum, training **tidak diulang**.

Per-run disimpan:

- `best_model.keras`
- `latest.weights.h5`
- `history.csv`
- `preds.npz`
- `run_state.json`
- `early_stopping_state.json`
- `training_complete.json`
- `backup/` selama training terputus/aktif

Notebook tidak menyimpan credential Kaggle.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os, sys, json, glob, re, shutil, random, gc, datetime
import hashlib, platform, subprocess, traceback

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, brier_score_loss
)

PROJECT_NAME = "malaria_colorspace"
DRIVE_ROOT = Path("/content/drive/MyDrive")
PROJ_DIR = DRIVE_ROOT / PROJECT_NAME

SPLIT_DIR = PROJ_DIR / "splits"
ORIGINAL_RESULTS_CSV = PROJ_DIR / "results" / "all_results.csv"
DATASET_ZIP = PROJ_DIR / "cell_images.zip"
LOCAL_DATA = Path("/content/data")

CANON_DIR = PROJ_DIR / "canonical_preprocessing"
ART_DIR = CANON_DIR / "artifacts"
RESULTS_DIR = CANON_DIR / "results"
PROV_DIR = CANON_DIR / "provenance"
COMPARE_DIR = CANON_DIR / "comparison"

LEDGER_PATH = CANON_DIR / "run_ledger_canonical.json"
RESULTS_CSV = RESULTS_DIR / "canonical_results.csv"
CONFIG_PATH = CANON_DIR / "canonical_config.json"

for d in [CANON_DIR, ART_DIR, RESULTS_DIR, PROV_DIR, COMPARE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

assert SPLIT_DIR.exists(), f"Split directory missing: {SPLIT_DIR}"
assert ORIGINAL_RESULTS_CSV.exists(), f"Original results missing: {ORIGINAL_RESULTS_CSV}"
assert DATASET_ZIP.exists(), (
    f"Dataset ZIP missing: {DATASET_ZIP}. "
    "Place the public dataset ZIP there; do not paste private Kaggle credentials."
)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))
print("Canonical root:", CANON_DIR)

In [ ]:
IMG_SIZE = 224
BATCH = 32
MAX_EPOCHS = 50
LR_FROZEN = 1e-4
LR_FINETUNE = 1e-5
FINETUNE_UNFREEZE_FROM = 100
PATIENCE = 5

N_SPLITS = 5
SPLIT_SEEDS = [42, 123, 7, 2024, 99]
TRAIN_SEED = 42
POS_LABEL = 1

PREPROCESSING = "mobilenet_v2_canonical"
EXPERIMENT_TAG = "exp5_canonical"
GRAY_WEIGHTS = tf.constant([0.2989, 0.5870, 0.1140], dtype=tf.float32)

EXPECTED_SPLIT_SHA256 = {
    "split_0_seed42.csv": "b18ed348f559c2f90f5cc71806c5302296bea7af6490f08dbe3532c0ca3241a6",
    "split_1_seed123.csv": "9698fae7fa62a191846dfe6196af6a745c8dc6af5001d9dbca6e5a30efeaec41",
    "split_2_seed7.csv": "0719562d2623fd3602cd7f80ca174bd2bb4962eaf314caabcbb6ceb2cdd3352f",
    "split_3_seed2024.csv": "1fbe3b529d8359e240d4fb2294cc7dccb8152c529469fdcf8823350a49321f86",
    "split_4_seed99.csv": "9dee1e6abb0e17b20b8c0e12861584a3b9ca9ba080aa5c44ec229d7fab028673",
}

CONFIG = {
    "experiment_tag": EXPERIMENT_TAG,
    "purpose": "Canonical MobileNetV2 preprocessing sensitivity analysis",
    "preprocessing": PREPROCESSING,
    "img_size": IMG_SIZE,
    "batch": BATCH,
    "max_epochs": MAX_EPOCHS,
    "lr_frozen": LR_FROZEN,
    "lr_finetune": LR_FINETUNE,
    "finetune_unfreeze_from": FINETUNE_UNFREEZE_FROM,
    "patience": PATIENCE,
    "n_splits": N_SPLITS,
    "split_seeds": SPLIT_SEEDS,
    "training_seed_fixed": TRAIN_SEED,
    "positive_label": POS_LABEL,
    "gray_weights": [0.2989, 0.5870, 0.1140],
    "backbone": "MobileNetV2",
    "backbone_weights": "imagenet",
    "backbone_call_training": False,
    "expected_new_runs": 20,
}

with open(CONFIG_PATH, "w") as f:
    json.dump(CONFIG, f, indent=2)

print(json.dumps(CONFIG, indent=2))

In [ ]:
def now_iso():
    return datetime.datetime.now(datetime.timezone.utc).isoformat()

def log(msg):
    print(f"[{datetime.datetime.now().strftime('%H:%M:%S')}] {msg}", flush=True)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)

def sha256_file(path, block_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(block_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def atomic_json_dump(obj, path):
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2)
    os.replace(tmp, path)

def atomic_csv_write(df, path):
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    df.to_csv(tmp, index=False)
    os.replace(tmp, path)

def load_json(path, default=None):
    path = Path(path)
    if not path.exists():
        return {} if default is None else default
    try:
        with open(path) as f:
            return json.load(f)
    except Exception:
        return {} if default is None else default

def load_ledger():
    return load_json(LEDGER_PATH, default={})

def mark_ledger(run_id, status, extra=None):
    ledger = load_ledger()
    entry = ledger.get(run_id, {})
    entry.update({
        "run_id": run_id,
        "status": status,
        "updated_at_utc": now_iso(),
    })
    if extra:
        entry.update(extra)
    ledger[run_id] = entry
    atomic_json_dump(ledger, LEDGER_PATH)

def update_run_state(run_dir, **kwargs):
    path = Path(run_dir) / "run_state.json"
    state = load_json(path, default={})
    state.update(kwargs)
    state["updated_at_utc"] = now_iso()
    atomic_json_dump(state, path)
    return state

def read_results():
    return pd.read_csv(RESULTS_CSV) if RESULTS_CSV.exists() else pd.DataFrame()

def upsert_result(row):
    df = read_results()
    new = pd.DataFrame([row])
    if len(df):
        df = df[df["run_id"] != row["run_id"]]
        df = pd.concat([df, new], ignore_index=True)
    else:
        df = new
    df = df.sort_values(["split", "scheme", "finetune", "run_id"]).reset_index(drop=True)
    atomic_csv_write(df, RESULTS_CSV)

def artifact_paths(run_id):
    run_dir = ART_DIR / run_id
    return {
        "run_dir": run_dir,
        "best_model": run_dir / "best_model.keras",
        "latest_weights": run_dir / "latest.weights.h5",
        "history": run_dir / "history.csv",
        "preds": run_dir / "preds.npz",
        "state": run_dir / "run_state.json",
        "early_stop": run_dir / "early_stopping_state.json",
        "training_complete": run_dir / "training_complete.json",
        "backup": run_dir / "backup",
    }

def run_is_complete(run_id):
    p = artifact_paths(run_id)
    required = [
        p["best_model"], p["latest_weights"], p["history"],
        p["preds"], p["training_complete"]
    ]
    if not all(x.exists() for x in required):
        return False

    df = read_results()
    if len(df) == 0 or run_id not in set(df["run_id"]):
        return False

    try:
        with np.load(p["preds"], allow_pickle=False) as d:
            required_arrays = {"y_val", "p_val", "y_test", "p_test"}
            if not required_arrays.issubset(d.files):
                return False
            if len(d["y_test"]) != len(d["p_test"]) or len(d["y_test"]) == 0:
                return False
            if not np.isfinite(d["p_test"]).all():
                return False
    except Exception:
        return False

    return True

## Runtime provenance

Versi runtime, GPU, `nvidia-smi`, dan `pip freeze` disimpan ke Drive **sebelum training**.

In [ ]:
import sklearn, scipy, matplotlib

env_lines = [
    f"timestamp_utc={now_iso()}",
    f"python={sys.version}",
    f"platform={platform.platform()}",
    f"tensorflow={tf.__version__}",
    f"numpy={np.__version__}",
    f"pandas={pd.__version__}",
    f"scikit-learn={sklearn.__version__}",
    f"scipy={scipy.__version__}",
    f"matplotlib={matplotlib.__version__}",
    f"gpus={tf.config.list_physical_devices('GPU')}",
]

try:
    smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=False)
    env_lines.append("\n--- nvidia-smi ---\n" + smi.stdout + smi.stderr)
except Exception as e:
    env_lines.append(f"nvidia-smi_error={repr(e)}")

(PROV_DIR / "environment_runtime.txt").write_text("\n".join(env_lines))

freeze = subprocess.run(
    [sys.executable, "-m", "pip", "freeze"],
    capture_output=True, text=True, check=False
)
(PROV_DIR / "pip_freeze.txt").write_text(freeze.stdout)

print("\n".join(env_lines[:10]))

## Local dataset reconstruction

ZIP tetap di Drive. Citra diekstrak ke `/content/data` agar training lebih cepat. Jika runtime reset, cell ini membangun ulang cache lokal secara otomatis.

In [ ]:
def prepare_local_data():
    if LOCAL_DATA.exists():
        existing = next(LOCAL_DATA.rglob("*.png"), None)
        if existing is not None:
            log("Local image cache already exists.")
            return

    assert DATASET_ZIP.exists(), f"Missing dataset ZIP: {DATASET_ZIP}"

    if LOCAL_DATA.exists():
        shutil.rmtree(LOCAL_DATA)
    LOCAL_DATA.mkdir(parents=True, exist_ok=True)

    log(f"Extracting {DATASET_ZIP} -> {LOCAL_DATA}")
    shutil.unpack_archive(str(DATASET_ZIP), str(LOCAL_DATA))
    log("Extraction completed.")

prepare_local_data()

def find_class_dirs(root):
    par = un = None
    for dp, dn, fn in os.walk(root):
        b = os.path.basename(dp).lower()
        if b == "parasitized":
            par = Path(dp)
        elif b == "uninfected":
            un = Path(dp)
    return par, un

par_dir, un_dir = find_class_dirs(LOCAL_DATA)
assert par_dir is not None and un_dir is not None

n_par = len(list(par_dir.glob("*.png")))
n_un = len(list(un_dir.glob("*.png")))
print("Parasitized:", n_par, "| Uninfected:", n_un, "| Total:", n_par + n_un)
assert n_par + n_un == 27558, "Unexpected dataset size. Stop before training."

## Frozen split integrity gate

Notebook **menolak training** jika salah satu split CSV berbeda dari file asli yang sudah diaudit sebelumnya.

In [ ]:
split_files = sorted(SPLIT_DIR.glob("split_*.csv"))
assert len(split_files) == 5

rows = []
for split_i, f in enumerate(split_files):
    actual = sha256_file(f)
    expected = EXPECTED_SPLIT_SHA256.get(f.name)
    assert expected is not None, f"Unexpected split filename: {f.name}"
    assert actual == expected, (
        f"Frozen split changed: {f.name}\nexpected={expected}\nactual={actual}"
    )

    s = pd.read_csv(f)
    assert {"filepath", "label", "group", "subset"}.issubset(s.columns)
    assert len(s) == 27558

    tr = s[s.subset == "train"]
    va = s[s.subset == "val"]
    te = s[s.subset == "test"]

    overlaps = {
        "group_train_val": len(set(tr.group) & set(va.group)),
        "group_train_test": len(set(tr.group) & set(te.group)),
        "group_val_test": len(set(va.group) & set(te.group)),
        "path_train_val": len(set(tr.filepath) & set(va.filepath)),
        "path_train_test": len(set(tr.filepath) & set(te.filepath)),
        "path_val_test": len(set(va.filepath) & set(te.filepath)),
    }
    assert max(overlaps.values()) == 0

    missing_paths = sum(not Path(p).exists() for p in s.filepath)
    assert missing_paths == 0, (
        f"{f.name}: {missing_paths} local image paths missing. "
        "Re-run extraction."
    )

    rows.append({
        "split": split_i,
        "seed": SPLIT_SEEDS[split_i],
        "file": f.name,
        "sha256": actual,
        "n_train": len(tr),
        "n_val": len(va),
        "n_test": len(te),
        "train_groups": tr.group.nunique(),
        "val_groups": va.group.nunique(),
        "test_groups": te.group.nunique(),
        **overlaps,
        "missing_local_paths": missing_paths,
        "status": "PASS",
    })

split_integrity = pd.DataFrame(rows)
display(split_integrity)
atomic_csv_write(split_integrity, PROV_DIR / "split_integrity_preflight.csv")
print("PASS: exact frozen split files verified.")

In [ ]:
def load_split(i):
    s = pd.read_csv(split_files[i])
    return (
        s[s.subset == "train"].reset_index(drop=True),
        s[s.subset == "val"].reset_index(drop=True),
        s[s.subset == "test"].reset_index(drop=True),
    )

## Canonical input pipeline

Urutan dipertahankan sesuai eksperimen asli:

`decode 0..255 -> grayscale bila perlu -> augmentation bila training -> clip 0..255 -> MobileNetV2 preprocess_input`

**Jangan melakukan `/255.0` sebelum `preprocess_input`.**

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def decode(path):
    img = tf.io.decode_png(tf.io.read_file(path), channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    return tf.cast(img, tf.float32)

def to_scheme(img, scheme):
    if scheme == "grayscale":
        g = tf.tensordot(img, GRAY_WEIGHTS, axes=[[-1], [0]])
        img = tf.stack([g, g, g], axis=-1)
    elif scheme != "rgb":
        raise ValueError(f"Unknown scheme: {scheme}")
    return img

def augment(img):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, max_delta=0.1 * 255.0)
    img = tf.image.resize_with_crop_or_pad(img, IMG_SIZE + 20, IMG_SIZE + 20)
    img = tf.image.random_crop(img, [IMG_SIZE, IMG_SIZE, 3])
    return img

def canonical_preprocess(img):
    img = tf.clip_by_value(img, 0.0, 255.0)
    return tf.keras.applications.mobilenet_v2.preprocess_input(img)

def make_ds(sub_df, scheme, training, batch=BATCH, shuffle_seed=TRAIN_SEED):
    paths = sub_df.filepath.values
    labels = sub_df.label.values.astype("int32")

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(
            len(paths),
            seed=shuffle_seed,
            reshuffle_each_iteration=True
        )

    def _map(path, label):
        img = decode(path)
        img = to_scheme(img, scheme)
        if training:
            img = augment(img)
        img = canonical_preprocess(img)
        return img, label

    ds = ds.map(_map, num_parallel_calls=AUTOTUNE, deterministic=True)
    return ds.batch(batch).prefetch(AUTOTUNE)

In [ ]:
tr0, va0, te0 = load_split(0)

rgb_x, _ = next(iter(make_ds(te0.head(4), "rgb", False, batch=4)))
gray_x, _ = next(iter(make_ds(te0.head(4), "grayscale", False, batch=4)))

print("RGB range:", float(tf.reduce_min(rgb_x)), float(tf.reduce_max(rgb_x)))
print("Gray range:", float(tf.reduce_min(gray_x)), float(tf.reduce_max(gray_x)))

assert float(tf.reduce_min(rgb_x)) >= -1.01
assert float(tf.reduce_max(rgb_x)) <= 1.01
assert float(tf.reduce_min(gray_x)) >= -1.01
assert float(tf.reduce_max(gray_x)) <= 1.01

d01 = float(tf.reduce_max(tf.abs(gray_x[...,0] - gray_x[...,1])))
d12 = float(tf.reduce_max(tf.abs(gray_x[...,1] - gray_x[...,2])))
assert d01 < 1e-6 and d12 < 1e-6

print("PASS: canonical preprocessing and grayscale replication.")

## Model definition

Arsitektur dibuat identik dengan eksperimen asli.

In [ ]:
def build_model(finetune=False, seed=TRAIN_SEED):
    set_seed(seed)

    base = tf.keras.applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights="imagenet"
    )

    if finetune:
        base.trainable = True
        for layer in base.layers[:FINETUNE_UNFREEZE_FROM]:
            layer.trainable = False
    else:
        base.trainable = False

    inp = tf.keras.Input((IMG_SIZE, IMG_SIZE, 3))
    x = base(inp, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(256, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    x = tf.keras.layers.Dense(128, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    out = tf.keras.layers.Dense(2, activation="softmax")(x)

    model = tf.keras.Model(inp, out)

    lr = LR_FINETUNE if finetune else LR_FROZEN
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

m = build_model(False)
print("Parameter count:", m.count_params())
assert m.count_params() == 2619074
del m
tf.keras.backend.clear_session()

## Persistent early stopping

State `best`, `wait`, dan `best_epoch` disimpan ke Drive setiap epoch, sehingga patience tidak kembali ke nol hanya karena runtime terputus.

Best model tetap ditentukan oleh `ModelCheckpoint(... monitor="val_loss", save_best_only=True)`.

In [ ]:
class PersistentEarlyStopping(tf.keras.callbacks.Callback):
    def __init__(self, state_path, monitor="val_loss", patience=5, min_delta=0.0):
        super().__init__()
        self.state_path = Path(state_path)
        self.monitor = monitor
        self.patience = patience
        self.min_delta = min_delta
        self.best = np.inf
        self.wait = 0
        self.best_epoch = -1

    def on_train_begin(self, logs=None):
        state = load_json(self.state_path, default={})
        self.best = float(state.get("best", np.inf))
        self.wait = int(state.get("wait", 0))
        self.best_epoch = int(state.get("best_epoch", -1))
        log(
            f"EarlyStop restore: best={self.best}, "
            f"wait={self.wait}, best_epoch={self.best_epoch}"
        )

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        current = logs.get(self.monitor)
        if current is None:
            return

        current = float(current)
        if current < self.best - self.min_delta:
            self.best = current
            self.wait = 0
            self.best_epoch = int(epoch)
        else:
            self.wait += 1

        atomic_json_dump({
            "monitor": self.monitor,
            "best": float(self.best),
            "wait": int(self.wait),
            "best_epoch": int(self.best_epoch),
            "patience": int(self.patience),
            "updated_at_utc": now_iso(),
        }, self.state_path)

        if self.wait >= self.patience:
            log(
                f"Persistent early stopping at epoch {epoch + 1}; "
                f"best epoch={self.best_epoch + 1}"
            )
            self.model.stop_training = True

## Resume-safe training/evaluation engine

Sebuah run baru dianggap `done` **setelah** model, weights, history, predictions, dan metrics semuanya tersimpan.

In [ ]:
def compute_metrics(y, p):
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    yhat = (p >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0,1]).ravel()
    return {
        "accuracy": accuracy_score(y, yhat),
        "precision": precision_score(y, yhat, pos_label=POS_LABEL, zero_division=0),
        "recall": recall_score(y, yhat, pos_label=POS_LABEL, zero_division=0),
        "f1": f1_score(y, yhat, pos_label=POS_LABEL, zero_division=0),
        "auc": roc_auc_score(y, p),
        "brier": brier_score_loss((y == POS_LABEL).astype(int), p),
        "n_test": len(y),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

def clean_history_csv(path):
    path = Path(path)
    if not path.exists():
        return
    try:
        h = pd.read_csv(path)
        if "epoch" in h.columns:
            h = h.drop_duplicates("epoch", keep="last").sort_values("epoch")
            atomic_csv_write(h, path)
    except Exception as e:
        log(f"History cleanup warning: {e}")

def run_one_canonical(split_i, scheme, finetune):
    ft = "ft" if finetune else "fr"
    run_id = f"{EXPERIMENT_TAG}_{scheme}_{ft}_split{split_i}"
    p = artifact_paths(run_id)
    p["run_dir"].mkdir(parents=True, exist_ok=True)

    if run_is_complete(run_id):
        mark_ledger(run_id, "done", {"reconciled": True})
        log(f"SKIP {run_id}: already scientifically complete.")
        return

    meta = {
        "split": split_i,
        "split_seed": SPLIT_SEEDS[split_i],
        "scheme": scheme,
        "finetune": bool(finetune),
        "train_seed": TRAIN_SEED,
        "preprocessing": PREPROCESSING,
    }
    mark_ledger(run_id, "running", meta)
    update_run_state(
        p["run_dir"],
        run_id=run_id,
        status="running",
        phase="initializing",
        **meta
    )

    try:
        tr, va, te = load_split(split_i)

        train_ds = make_ds(tr, scheme, True, shuffle_seed=TRAIN_SEED)
        val_ds = make_ds(va, scheme, False)
        test_ds = make_ds(te, scheme, False)

        if p["training_complete"].exists() and p["best_model"].exists():
            log(f"RECOVER {run_id}: training complete; continue evaluation only.")
            update_run_state(p["run_dir"], phase="evaluation_recovery")
            model = tf.keras.models.load_model(p["best_model"])

        else:
            update_run_state(p["run_dir"], phase="training")
            model = build_model(finetune=finetune, seed=TRAIN_SEED)

            backup_exists = p["backup"].exists() and any(p["backup"].rglob("*"))
            initial_epoch = 0

            if backup_exists:
                log(
                    f"PRIMARY RESUME {run_id}: Keras BackupAndRestore "
                    "checkpoint found on Drive."
                )
            elif p["latest_weights"].exists() and p["history"].exists():
                try:
                    old_hist = pd.read_csv(p["history"])
                    if len(old_hist):
                        initial_epoch = int(old_hist["epoch"].max()) + 1
                        model.load_weights(p["latest_weights"])
                        log(
                            f"FALLBACK RESUME {run_id}: "
                            f"initial_epoch={initial_epoch}"
                        )
                except Exception as e:
                    log(f"Fallback resume failed; restart epoch 0: {e}")
                    initial_epoch = 0

            callbacks = [
                tf.keras.callbacks.BackupAndRestore(
                    backup_dir=str(p["backup"]),
                    save_freq="epoch",
                    delete_checkpoint=True,
                ),
                tf.keras.callbacks.ModelCheckpoint(
                    str(p["best_model"]),
                    monitor="val_loss",
                    save_best_only=True,
                    save_weights_only=False,
                    verbose=1,
                ),
                tf.keras.callbacks.ModelCheckpoint(
                    str(p["latest_weights"]),
                    save_best_only=False,
                    save_weights_only=True,
                    save_freq="epoch",
                    verbose=0,
                ),
                tf.keras.callbacks.CSVLogger(
                    str(p["history"]),
                    append=p["history"].exists(),
                ),
                PersistentEarlyStopping(
                    p["early_stop"],
                    monitor="val_loss",
                    patience=PATIENCE,
                ),
            ]

            fit_started = now_iso()
            model.fit(
                train_ds,
                validation_data=val_ds,
                epochs=MAX_EPOCHS,
                initial_epoch=initial_epoch,
                callbacks=callbacks,
                verbose=1,
            )
            fit_finished = now_iso()

            clean_history_csv(p["history"])

            assert p["best_model"].exists()
            assert p["latest_weights"].exists()
            assert p["history"].exists()

            hist = pd.read_csv(p["history"])
            atomic_json_dump({
                "run_id": run_id,
                "training_complete": True,
                "fit_started_utc": fit_started,
                "fit_finished_utc": fit_finished,
                "epochs_logged": int(len(hist)),
                "last_epoch_logged": int(hist["epoch"].max()) if len(hist) else None,
                "best_model_sha256": sha256_file(p["best_model"]),
                "latest_weights_sha256": sha256_file(p["latest_weights"]),
            }, p["training_complete"])

            update_run_state(
                p["run_dir"],
                phase="training_complete",
                training_complete=True
            )

            del model
            tf.keras.backend.clear_session()
            gc.collect()
            model = tf.keras.models.load_model(p["best_model"])

        update_run_state(p["run_dir"], phase="evaluating")

        p_val = model.predict(val_ds, verbose=0)[:, POS_LABEL]
        p_test = model.predict(test_ds, verbose=0)[:, POS_LABEL]
        y_val = va.label.values.astype(int)
        y_test = te.label.values.astype(int)

        assert len(y_val) == len(p_val)
        assert len(y_test) == len(p_test)
        assert np.isfinite(p_val).all() and np.isfinite(p_test).all()
        assert ((p_val >= 0) & (p_val <= 1)).all()
        assert ((p_test >= 0) & (p_test <= 1)).all()

        tmp_preds = p["run_dir"] / "preds.tmp.npz"
        np.savez(
            tmp_preds,
            y_val=y_val, p_val=p_val,
            y_test=y_test, p_test=p_test
        )
        os.replace(tmp_preds, p["preds"])

        metrics = compute_metrics(y_test, p_test)

        row = {
            "run_id": run_id,
            "tag": EXPERIMENT_TAG,
            "preprocessing": PREPROCESSING,
            "split": split_i,
            "split_seed": SPLIT_SEEDS[split_i],
            "scheme": scheme,
            "finetune": bool(finetune),
            "train_seed": TRAIN_SEED,
            **metrics,
            "best_model_sha256": sha256_file(p["best_model"]),
            "latest_weights_sha256": sha256_file(p["latest_weights"]),
            "preds_sha256": sha256_file(p["preds"]),
            "completed_at_utc": now_iso(),
        }

        upsert_result(row)

        update_run_state(
            p["run_dir"],
            status="done",
            phase="done",
            metrics={k: float(row[k]) for k in
                     ["accuracy","precision","recall","f1","auc","brier"]}
        )

        mark_ledger(run_id, "done", {
            **meta,
            "metrics": {
                k: float(row[k]) for k in
                ["accuracy","precision","recall","f1","auc","brier"]
            },
            "artifacts": {
                "best_model": str(p["best_model"]),
                "latest_weights": str(p["latest_weights"]),
                "history": str(p["history"]),
                "preds": str(p["preds"]),
            },
        })

        log(
            f"DONE {run_id}: acc={row['accuracy']:.4f} "
            f"auc={row['auc']:.4f} recall={row['recall']:.4f} fn={row['fn']}"
        )

        del model
        tf.keras.backend.clear_session()
        gc.collect()

    except Exception as e:
        tb = traceback.format_exc()
        update_run_state(
            p["run_dir"],
            status="interrupted_or_failed",
            phase="error",
            error=repr(e),
            traceback=tb,
        )
        mark_ledger(run_id, "interrupted_or_failed", {
            **meta,
            "error": repr(e),
        })
        print(tb)
        raise

## Reconnect runbook disimpan otomatis ke Drive

In [ ]:
runbook = f"""
# Canonical preprocessing experiment - reconnect/recovery runbook

Experiment: {EXPERIMENT_TAG}
Updated UTC: {now_iso()}

Persistent root:
{CANON_DIR}

After Colab disconnect:
1. Reconnect to a GPU runtime.
2. Open this notebook and run from the top.
3. Mount Google Drive.
4. Local /content/data will be reconstructed from the ZIP if needed.
5. Frozen splits will be SHA-256 verified.
6. Run the 20-run experiment loop again.
7. Completed runs are skipped.
8. Interrupted runs attempt recovery from artifacts/<run_id>/backup/.
9. If backup is unavailable, latest.weights.h5 + history.csv are fallback.
10. If training_complete.json exists, training is not repeated; evaluation resumes.

Never delete during active experiment:
- {ART_DIR}
- {LEDGER_PATH}
- {RESULTS_CSV}

Expected new runs: 20
Fixed training seed: {TRAIN_SEED}
Split seeds: {SPLIT_SEEDS}

Canonical transform:
tf.keras.applications.mobilenet_v2.preprocess_input(x)

Input must still be in the 0..255 scale before preprocess_input.
Do not divide by 255 first.

Scientific note:
Saved model/optimizer state, early-stopping state, logs, predictions, split
assignments and seeds are persistent. Exact bitwise GPU execution and tf.data
iterator state are not guaranteed across different Colab physical runtimes.
"""
(CANON_DIR / "RUNBOOK_RECONNECT.md").write_text(runbook)
print(runbook)

# Run 20 canonical experiments

Urutan: setiap split menyelesaikan 4 kondisi sebelum pindah ke split berikutnya.

Setelah reconnect, rerun cell ini. Run yang sudah lengkap otomatis `SKIP`.

In [ ]:
RUN_EXPERIMENTS = True

if RUN_EXPERIMENTS:
    for split_i in range(N_SPLITS):
        for scheme in ["grayscale", "rgb"]:
            for finetune in [False, True]:
                run_one_canonical(split_i, scheme, finetune)
else:
    print("RUN_EXPERIMENTS=False")

# Progress audit

Aman dijalankan kapan saja.

In [ ]:
expected_run_ids = [
    f"{EXPERIMENT_TAG}_{scheme}_{'ft' if ft else 'fr'}_split{split_i}"
    for split_i in range(N_SPLITS)
    for scheme in ["grayscale", "rgb"]
    for ft in [False, True]
]

rows = []
for run_id in expected_run_ids:
    p = artifact_paths(run_id)
    ledger_status = load_ledger().get(run_id, {}).get("status", "not_started")
    rows.append({
        "run_id": run_id,
        "ledger_status": ledger_status,
        "best_model": p["best_model"].exists(),
        "latest_weights": p["latest_weights"].exists(),
        "history": p["history"].exists(),
        "preds": p["preds"].exists(),
        "training_complete": p["training_complete"].exists(),
        "scientifically_complete": run_is_complete(run_id),
    })

progress_df = pd.DataFrame(rows)
display(progress_df)
atomic_csv_write(progress_df, RESULTS_DIR / "canonical_progress_audit.csv")

n_complete = int(progress_df.scientifically_complete.sum())
print(f"{n_complete}/20 canonical runs scientifically complete")

# Mandatory 20/20 prediction integrity audit

Metrics dihitung ulang langsung dari 20 `preds.npz`.

In [ ]:
canonical_results = read_results()

assert len(canonical_results) == 20, (
    f"Expected 20 canonical rows, found {len(canonical_results)}"
)
assert canonical_results.run_id.nunique() == 20

missing = [r for r in expected_run_ids if not run_is_complete(r)]
assert not missing, f"Incomplete runs: {missing}"

audit_rows = []
for _, ref in canonical_results.iterrows():
    p = artifact_paths(ref.run_id)
    with np.load(p["preds"], allow_pickle=False) as d:
        y_test = d["y_test"]
        p_test = d["p_test"]
        y_val = d["y_val"]
        p_val = d["p_val"]

    rec = compute_metrics(y_test, p_test)
    max_diff = max(
        abs(float(rec[m]) - float(ref[m]))
        for m in ["accuracy","precision","recall","f1","auc","brier"]
    )
    cm_match = all(
        int(rec[c]) == int(ref[c])
        for c in ["tn","fp","fn","tp"]
    )

    audit_rows.append({
        "run_id": ref.run_id,
        "n_test": len(y_test),
        "n_val": len(y_val),
        "finite_test_prob": bool(np.isfinite(p_test).all()),
        "finite_val_prob": bool(np.isfinite(p_val).all()),
        "max_metric_absdiff": max_diff,
        "confusion_exact_match": cm_match,
        "status": "PASS" if max_diff <= 1e-6 and cm_match else "FAIL",
    })

canonical_integrity = pd.DataFrame(audit_rows)
display(canonical_integrity)
atomic_csv_write(
    canonical_integrity,
    RESULTS_DIR / "canonical_prediction_integrity_audit_20runs.csv"
)
assert (canonical_integrity.status == "PASS").all()
print("PASS: 20/20 canonical prediction archives reproduce stored metrics.")

# Canonical-only summary

In [ ]:
canonical_summary = (
    canonical_results
    .groupby(["scheme", "finetune"])
    .agg(
        n=("run_id","count"),
        accuracy_mean=("accuracy","mean"),
        accuracy_sd=("accuracy","std"),
        precision_mean=("precision","mean"),
        precision_sd=("precision","std"),
        recall_mean=("recall","mean"),
        recall_sd=("recall","std"),
        f1_mean=("f1","mean"),
        f1_sd=("f1","std"),
        auc_mean=("auc","mean"),
        auc_sd=("auc","std"),
        brier_mean=("brier","mean"),
        brier_sd=("brier","std"),
    )
    .reset_index()
)
display(canonical_summary)
atomic_csv_write(canonical_summary, RESULTS_DIR / "canonical_summary.csv")

# Direct comparison with original 20 runs

Menghasilkan combined **40 main runs** dengan faktor preprocessing eksplisit.

In [ ]:
old_all = pd.read_csv(ORIGINAL_RESULTS_CSV)
old_main = old_all[old_all.tag == "exp1"].copy()
assert len(old_main) == 20

old_main["preprocessing"] = "unit_0_1"
old_main["split_seed"] = old_main["split"].map(dict(enumerate(SPLIT_SEEDS)))

new_main = canonical_results.copy()

common_cols = [
    "run_id","tag","preprocessing","split","split_seed",
    "scheme","finetune","train_seed",
    "accuracy","precision","recall","f1","auc","brier",
    "n_test","tn","fp","fn","tp"
]

combined_40 = pd.concat(
    [old_main[common_cols], new_main[common_cols]],
    ignore_index=True
)
assert len(combined_40) == 40

display(combined_40.head())
atomic_csv_write(combined_40, COMPARE_DIR / "combined_main_40runs.csv")

In [ ]:
factorial_summary = (
    combined_40
    .groupby(["preprocessing","scheme","finetune"])
    .agg(
        n=("run_id","count"),
        accuracy_mean=("accuracy","mean"),
        accuracy_sd=("accuracy","std"),
        precision_mean=("precision","mean"),
        precision_sd=("precision","std"),
        recall_mean=("recall","mean"),
        recall_sd=("recall","std"),
        f1_mean=("f1","mean"),
        f1_sd=("f1","std"),
        auc_mean=("auc","mean"),
        auc_sd=("auc","std"),
        brier_mean=("brier","mean"),
        brier_sd=("brier","std"),
    )
    .reset_index()
)
display(factorial_summary)
atomic_csv_write(
    factorial_summary,
    COMPARE_DIR / "factorial_2x2x2_summary.csv"
)

## Paired canonical-minus-original differences by split

In [ ]:
old_key = old_main.set_index(["split","scheme","finetune"])
new_key = new_main.set_index(["split","scheme","finetune"])

rows = []
for key in new_key.index:
    n = new_key.loc[key]
    o = old_key.loc[key]
    split_i, scheme, ft = key

    row = {
        "split": int(split_i),
        "split_seed": SPLIT_SEEDS[int(split_i)],
        "scheme": scheme,
        "finetune": bool(ft),
    }

    for m in ["accuracy","precision","recall","f1","auc","brier"]:
        row[f"old_{m}"] = float(o[m])
        row[f"canonical_{m}"] = float(n[m])
        row[f"delta_canonical_minus_old_{m}"] = float(n[m] - o[m])

    rows.append(row)

paired_preprocessing = pd.DataFrame(rows)
display(paired_preprocessing)
atomic_csv_write(
    paired_preprocessing,
    COMPARE_DIR / "canonical_minus_original_per_split.csv"
)

summary_rows = []
for (scheme, ft), g in paired_preprocessing.groupby(["scheme","finetune"]):
    row = {"scheme": scheme, "finetune": bool(ft), "n": len(g)}
    for m in ["accuracy","precision","recall","f1","auc","brier"]:
        d = g[f"delta_canonical_minus_old_{m}"]
        row[f"delta_{m}_mean"] = d.mean()
        row[f"delta_{m}_sd"] = d.std(ddof=1)
    summary_rows.append(row)

paired_preprocessing_summary = pd.DataFrame(summary_rows)
display(paired_preprocessing_summary)
atomic_csv_write(
    paired_preprocessing_summary,
    COMPARE_DIR / "canonical_minus_original_summary.csv"
)

## RGB-minus-grayscale gap under both preprocessing pipelines

In [ ]:
rows = []

for prep in ["unit_0_1", "mobilenet_v2_canonical"]:
    dprep = combined_40[combined_40.preprocessing == prep]

    for ft in [False, True]:
        sub = dprep[dprep.finetune == ft]
        rgb = sub[sub.scheme == "rgb"].set_index("split")
        gray = sub[sub.scheme == "grayscale"].set_index("split")

        for split_i in range(N_SPLITS):
            row = {
                "preprocessing": prep,
                "finetune": ft,
                "split": split_i,
                "split_seed": SPLIT_SEEDS[split_i],
            }
            for m in ["accuracy","precision","recall","f1","auc","brier"]:
                row[f"rgb_minus_gray_{m}"] = (
                    float(rgb.loc[split_i, m]) -
                    float(gray.loc[split_i, m])
                )
            rows.append(row)

rgb_gray_gap = pd.DataFrame(rows)
display(rgb_gray_gap)
atomic_csv_write(
    rgb_gray_gap,
    COMPARE_DIR / "rgb_minus_grayscale_gap_per_split.csv"
)

rgb_gray_gap_summary = (
    rgb_gray_gap
    .groupby(["preprocessing","finetune"])
    .agg(
        n=("split","count"),
        accuracy_gap_mean=("rgb_minus_gray_accuracy","mean"),
        accuracy_gap_sd=("rgb_minus_gray_accuracy","std"),
        recall_gap_mean=("rgb_minus_gray_recall","mean"),
        recall_gap_sd=("rgb_minus_gray_recall","std"),
        auc_gap_mean=("rgb_minus_gray_auc","mean"),
        auc_gap_sd=("rgb_minus_gray_auc","std"),
    )
    .reset_index()
)

display(rgb_gray_gap_summary)
atomic_csv_write(
    rgb_gray_gap_summary,
    COMPARE_DIR / "rgb_minus_grayscale_gap_summary.csv"
)

## Repeated-partition statistical sensitivity

Paired tests ini adalah evidence dari repeated partitions dataset yang sama, **bukan lima cohort independen**.

In [ ]:
from scipy import stats

rows = []

for scheme in ["grayscale","rgb"]:
    for ft in [False, True]:
        g = paired_preprocessing[
            (paired_preprocessing.scheme == scheme) &
            (paired_preprocessing.finetune == ft)
        ]

        for metric in ["accuracy","auc","recall"]:
            d = g[f"delta_canonical_minus_old_{metric}"].to_numpy()
            t, pval = stats.ttest_1samp(d, popmean=0.0)
            ci = stats.t.interval(
                0.95,
                len(d)-1,
                loc=d.mean(),
                scale=stats.sem(d)
            )

            rows.append({
                "contrast": "canonical_minus_unit_0_1",
                "scheme": scheme,
                "finetune": ft,
                "metric": metric,
                "n_repeated_partitions": len(d),
                "mean_difference": float(d.mean()),
                "ci_low": float(ci[0]),
                "ci_high": float(ci[1]),
                "t": float(t),
                "p": float(pval),
            })

preprocessing_stats = pd.DataFrame(rows)
display(preprocessing_stats)
atomic_csv_write(
    preprocessing_stats,
    COMPARE_DIR / "preprocessing_paired_stats.csv"
)

# Publication-support figures

Disimpan di `canonical_preprocessing/comparison/figures/`.

In [ ]:
import matplotlib.pyplot as plt

FIG_DIR = COMPARE_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

plot_df = factorial_summary.copy()
plot_df["label"] = (
    plot_df["preprocessing"].map({
        "unit_0_1": "[0,1]",
        "mobilenet_v2_canonical": "Canonical",
    })
    + " | " + plot_df["scheme"].str.upper()
    + " | " + np.where(plot_df["finetune"], "FT", "Frozen")
)

fig, ax = plt.subplots(figsize=(12,5))
x = np.arange(len(plot_df))
ax.bar(x, plot_df["accuracy_mean"])
ax.errorbar(
    x, plot_df["accuracy_mean"],
    yerr=plot_df["accuracy_sd"],
    fmt="none", capsize=4
)
ax.set_xticks(x)
ax.set_xticklabels(plot_df["label"], rotation=35, ha="right")
ax.set_ylabel("Accuracy")
ax.set_title("Original vs Canonical MobileNetV2 Preprocessing")
fig.tight_layout()
fig.savefig(FIG_DIR / "accuracy_original_vs_canonical.png", dpi=180)
plt.show()

gap_plot = rgb_gray_gap_summary.copy()
gap_plot["label"] = (
    gap_plot["preprocessing"].map({
        "unit_0_1": "[0,1]",
        "mobilenet_v2_canonical": "Canonical",
    })
    + " | " + np.where(gap_plot["finetune"], "FT", "Frozen")
)

fig, ax = plt.subplots(figsize=(8,5))
x = np.arange(len(gap_plot))
ax.bar(x, gap_plot["accuracy_gap_mean"])
ax.errorbar(
    x, gap_plot["accuracy_gap_mean"],
    yerr=gap_plot["accuracy_gap_sd"],
    fmt="none", capsize=4
)
ax.axhline(0, linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(gap_plot["label"], rotation=25, ha="right")
ax.set_ylabel("RGB - Grayscale Accuracy")
ax.set_title("RGB-Grayscale Gap by Preprocessing")
fig.tight_layout()
fig.savefig(
    FIG_DIR / "rgb_gray_accuracy_gap_by_preprocessing.png",
    dpi=180
)
plt.show()

# SHA-256 provenance manifest

Meng-hash output utama seluruh 20 run dan comparison files.

In [ ]:
manifest_rows = []

def add_file(path, category, run_id=None):
    path = Path(path)
    if path.exists() and path.is_file():
        manifest_rows.append({
            "category": category,
            "run_id": run_id,
            "path": str(path),
            "size_bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        })

add_file(CONFIG_PATH, "experiment_config")
add_file(LEDGER_PATH, "run_ledger")
add_file(RESULTS_CSV, "canonical_results")
add_file(CANON_DIR / "RUNBOOK_RECONNECT.md", "reconnect_runbook")

for f in split_files:
    add_file(f, "frozen_split")

for run_id in expected_run_ids:
    p = artifact_paths(run_id)
    add_file(p["best_model"], "best_model", run_id)
    add_file(p["latest_weights"], "latest_weights", run_id)
    add_file(p["history"], "history", run_id)
    add_file(p["preds"], "predictions", run_id)
    add_file(p["state"], "run_state", run_id)
    add_file(p["early_stop"], "early_stopping_state", run_id)
    add_file(p["training_complete"], "training_complete", run_id)

for f in COMPARE_DIR.rglob("*"):
    if f.is_file():
        add_file(f, "comparison_output")

manifest_df = pd.DataFrame(manifest_rows)
display(manifest_df.head(20))
atomic_csv_write(
    manifest_df,
    PROV_DIR / "canonical_sha256_manifest.csv"
)
print("Manifest entries:", len(manifest_df))

# Final canonical experiment verdict

PASS mensyaratkan:

- 20/20 new runs complete;
- 20/20 prediction integrity PASS;
- exact original split hashes verified;
- direct combined 40-run comparison exists.

In [ ]:
complete_count = sum(run_is_complete(r) for r in expected_run_ids)

canonical_integrity = pd.read_csv(
    RESULTS_DIR / "canonical_prediction_integrity_audit_20runs.csv"
)
integrity_pass = int((canonical_integrity.status == "PASS").sum())

combined_path = COMPARE_DIR / "combined_main_40runs.csv"
factorial_path = COMPARE_DIR / "factorial_2x2x2_summary.csv"

combined_count = (
    len(pd.read_csv(combined_path))
    if combined_path.exists()
    else 0
)

final_status = "PASS" if (
    complete_count == 20
    and integrity_pass == 20
    and combined_count == 40
    and factorial_path.exists()
) else "FAIL"

final_verdict = pd.DataFrame([{
    "timestamp_utc": now_iso(),
    "experiment": EXPERIMENT_TAG,
    "preprocessing": PREPROCESSING,
    "expected_new_runs": 20,
    "complete_new_runs": complete_count,
    "prediction_integrity_pass": integrity_pass,
    "frozen_split_sha256_verified": True,
    "group_leakage_detected": False,
    "filepath_leakage_detected": False,
    "combined_main_runs_expected": 40,
    "combined_main_runs_present": combined_count,
    "overall_status": final_status,
}])

display(final_verdict)
atomic_csv_write(
    final_verdict,
    RESULTS_DIR / "FINAL_CANONICAL_EXPERIMENT_VERDICT.csv"
)

print("=" * 76)
print("CANONICAL PREPROCESSING SENSITIVITY EXPERIMENT:", final_status)
print(f"- New canonical CNN runs complete: {complete_count}/20")
print(f"- Prediction integrity: {integrity_pass}/20")
print("- Frozen split SHA-256 verification: PASS")
print("- Group/filepath leakage preflight: PASS")
print(f"- Combined main experiment rows: {combined_count}/40")
print("=" * 76)

assert final_status == "PASS", (
    "Experiment not yet complete. Check progress audit and rerun after reconnect."
)

# Guardrail untuk penulisan paper

Eksperimen ini **tidak mengganti** hasil `[0,1]`.

Framing yang benar:

> Original experiments used `[0,1]` scaling. Because this differs from canonical MobileNetV2 ImageNet preprocessing, the identical 2×2 RGB/grayscale × frozen/fine-tuned design was repeated across the same five frozen group-aware partitions using `tf.keras.applications.mobilenet_v2.preprocess_input`.

Jika pola hasil konsisten, kita dapat menyatakan temuan robust terhadap preprocessing.

Jika berubah, preprocessing harus dilaporkan sebagai faktor metodologis yang berinteraksi dengan representation/training strategy.

Lima split tetap repeated partitions dari dataset yang sama, bukan cohort independen.